# SRΨ-v1.0 Baseline Training (Colab - Fixed)

**版本**: v1.0-Fixed
**仓库**: https://github.com/Mozilla2004/srpsi-colab-baseline

**修复内容**:
- ✅ 修复了所有导入错误
- ✅ 简化了项目结构，只包含 v1.0 必要文件
- ✅ 正确处理工作目录

**预期性能**: Energy Drift ~10.88

---

## Phase 1: 环境准备

In [ ]:
# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 安装依赖
!pip install torch torchvision numpy pyyaml tqdm tensorboard -q

In [ ]:
# 克隆代码库并设置环境
import os
import sys
import shutil

# 重置到安全目录
os.chdir('/content')

# 清理旧代码（如果存在）
if os.path.exists('/content/srpsi-colab-baseline'):
    shutil.rmtree('/content/srpsi-colab-baseline')
    print("✅ 旧代码已清理")

# 克隆最新代码
print("🔄 正在克隆仓库...")
!git clone https://github.com/Mozilla2004/srpsi-colab-baseline.git /content/srpsi-colab-baseline

# 切换到项目目录
os.chdir('/content/srpsi-colab-baseline')
sys.path.insert(0, '/content/srpsi-colab-baseline')

print(f"\n✅ 代码库已准备")
print(f"📁 当前目录: {os.getcwd()}")
print(f"\n📄 文件列表:")
!ls -la

In [ ]:
# 验证环境
print("🔍 验证导入...")

# 验证关键文件
import os
files_to_check = [
    'src/train.py',
    'src/models/__init__.py',
    'src/models/srpsi_engine_tiny.py',
    'src/datasets.py',
    'config/burgers.yaml'
]

for f in files_to_check:
    exists = os.path.exists(f)
    print(f"  {'✅' if exists else '❌'} {f}")

# 验证导入
try:
    from src.models.srpsi_engine_tiny import SRPsiEngineTiny
    print("\n✅ SRPsiEngineTiny 导入成功")
except Exception as e:
    print(f"\n❌ 导入失败: {e}")
    
# 检查 train.py 的导入
with open('src/train.py', 'r') as f:
    content = f.read()
    if 'from src.models.srpsi_engine_tiny import SRPsiEngineTiny' in content:
        print("✅ train.py 导入正确")
    else:
        print("❌ train.py 导入有问题")

## Phase 2: 数据准备

In [ ]:
# 生成数据
import numpy as np
from datetime import datetime

# 创建数据目录
DATA_DIR = '/content/drive/MyDrive/srpsi-colab-baseline/data'
os.makedirs(DATA_DIR, exist_ok=True)

DATA_PATH = os.path.join(DATA_DIR, 'burgers_1d_v1.npy')

if not os.path.exists(DATA_PATH):
    print("📊 生成 Burgers 1D 数据...")
    
    from src.data_gen import generate_burgers_1d
    
    data = generate_burgers_1d(
        num_samples=4800,  # 4000 train + 400 val + 400 test
        total_steps=48,     # 16 tin + 32 tout
        nx=128,
        dt=0.01,
        dx=0.01,
        nu=0.01,
        seed=42
    )
    
    np.save(DATA_PATH, data)
    print(f"\n✅ 数据已保存: {DATA_PATH}")
    print(f"   Shape: {data.shape}")
else:
    print(f"✅ 数据已存在: {DATA_PATH}")
    data = np.load(DATA_PATH)
    print(f"   Shape: {data.shape}")

## Phase 3: 训练

In [ ]:
# 配置训练
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# 创建输出目录
OUTPUT_DIR = f'/content/drive/MyDrive/srpsi-colab-baseline/runs/v1_baseline_{timestamp}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📁 输出目录: {OUTPUT_DIR}")
print(f"📊 数据路径: {DATA_PATH}")
print(f"\n🚀 准备开始训练 (80 epochs, 预计 60-90 分钟)...")

In [ ]:
# 执行训练 (这会花 60-90 分钟)
!python src/train.py \
    --config config/burgers.yaml \
    --model srpsi_engine \
    --data {DATA_PATH} \
    --epochs 80 \
    --output {OUTPUT_DIR}

## Phase 4: 结果分析

In [ ]:
# 加载最佳模型并测试
import torch
from src.train import create_model, load_config, validate
from src.utils import get_device
from src.datasets import create_dataloaders

# 加载配置
cfg = load_config('config/burgers.yaml')
device = get_device('cuda')

# 创建模型
model = create_model('srpsi_engine', cfg, device)

# 加载 checkpoint
checkpoint_path = f"{OUTPUT_DIR}/checkpoints/final.pt"
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"✅ 模型已加载: {checkpoint_path}")

# 创建数据加载器
train_loader, val_loader, test_loader = create_dataloaders(
    data_path=DATA_PATH,
    tin=16,
    tout=32,
    batch_size=32,
    num_train=4000,
    num_val=400,
    num_test=400,
    num_workers=0
)

# 测试
print("\n🧪 运行测试...")
test_metrics = validate(model, test_loader, cfg, device)

print("\n" + "="*60)
print(" "*15 + "FINAL RESULTS")
print("="*60)
print(f"Test Loss:      {test_metrics['val_loss']:.6f}")
print(f"Test MSE:       {test_metrics['val_rollout_mse']:.6f}")
print(f"Test Drift:     {test_metrics['val_energy_drift']:.6f}")
print()

# 与 v1.0 基线对比
print("🎯 与 v1.0 基线对比:")
baseline_drift = 10.88
if test_metrics['val_energy_drift'] < baseline_drift:
    print(f"   ✅ Energy Drift 改善: {baseline_drift:.2f} → {test_metrics['val_energy_drift']:.2f}")
    print(f"   改善幅度: {((baseline_drift - test_metrics['val_energy_drift']) / baseline_drift * 100):.1f}%")
else:
    print(f"   Energy Drift: {test_metrics['val_energy_drift']:.2f} (基线: {baseline_drift:.2f})")
    print(f"   差距: {((test_metrics['val_energy_drift'] - baseline_drift) / baseline_drift * 100):.1f}%")

## Phase 5: 保存结果

In [ ]:
# 保存测试结果
import json

results = {
    'model': 'SRΨ-v1.0 (Baseline)',
    'timestamp': datetime.now().isoformat(),
    'test_metrics': {
        'loss': float(test_metrics['val_loss']),
        'rollout_mse': float(test_metrics['val_rollout_mse']),
        'energy_drift': float(test_metrics['val_energy_drift'])
    },
    'baseline_comparison': {
        'v1_energy_drift': 10.88,
        'v1_momentum_drift': 18.49,
        'diff_vs_baseline': f"{((test_metrics['val_energy_drift'] - 10.88) / 10.88 * 100):.1f}%"
    },
    'trae_guidance': {
        'concept': 'Normalization as Field Ruler',
        'insight': 'Input standardization provides the geometric scale for field evolution'
    }
}

results_path = f"{OUTPUT_DIR}/test_results.json"
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"💾 结果已保存: {results_path}")
print()
print("="*60)
print(" "*15 + "TRAINING COMPLETE ✅")
print("="*60)
print()
print("🎓 下一步: 分析 v1.0 的成功要素，准备注入 v2.0 特性")